# Wisdom of Crowds — Sweets in a Jar

> *"In these democratic days, any investigation into the trustworthiness and peculiarities of popular judgments is of interest."*
>
> — Francis Galton, *Vox Populi*, Nature **75**:450 (1907)

A faithful replication of Galton's 1907 experiment, on 60 guesses of the number of sweets in a jar. The notebook runs identically:

- **Locally** with PySpark 4.0.0 (Path A).
- **On Databricks** with DBR 17.3 LTS / Spark 4.0.0, opened via Databricks Repos alongside `src/`.

All logic lives in `src/wisdom_of_crowds/`. This notebook is a narrative shell that calls it.

## 1 · Config

Every parameter that varies between environments lives here. If a value is left as `None`, the notebook falls back to a Databricks widget of the same name. On Databricks these constants are nulled automatically by the widget-bootstrap cell below; locally they are used as-is.

In [ ]:
# Local defaults. On Databricks the next cell nulls these and declares widgets
# with the same names so the resolver picks up the widget values.
INPUT_PATH:   str | None = "../data/sweets-jar-guesses.csv"
TRUE_COUNT:   int | None = None                 # Set once we count the jar.
OUTPUT_TABLE: str | None = "../output/summary"  # Delta on DBX, Parquet locally.

EXPERIMENT_ID: str = "sweets-jar-2026-09-19"

## 1a · Databricks widget bootstrap (no-op locally)

On Databricks this cell declares the three widgets and nulls the local defaults above, so the config resolver falls back to widget values. Locally, `dbutils` is not defined and the cell does nothing.

In [ ]:
# Databricks injects a `dbutils` global into the notebook's own namespace.
# It is NOT available inside `src/wisdom_of_crowds/config.py` (a different
# module), so we read the widget values here and assign them directly to
# the notebook-level constants that build_config() consumes below.
try:
    _dbx = dbutils  # type: ignore[name-defined]  # supplied by the Databricks runtime
except NameError:
    _dbx = None

if _dbx is not None:
    _dbx.widgets.text(
        "input_path",
        "/Volumes/workspace/default/wisdom_of_crowds/sweets-jar-guesses.csv",
        "Input CSV (Unity Catalog Volume path)",
    )
    _dbx.widgets.text("true_count", "", "True number of sweets (int)")
    _dbx.widgets.text(
        "output_table",
        "workspace.default.wisdom_of_crowds_summary",
        "Output Delta table (catalog.schema.table) or Volume path",
    )

    # Resolve the widget values and hand them straight to the notebook-level
    # constants. build_config() then receives concrete strings/int, no fallback.
    _ip = (_dbx.widgets.get("input_path") or "").strip()
    _tc = (_dbx.widgets.get("true_count") or "").strip()
    _ot = (_dbx.widgets.get("output_table") or "").strip()

    INPUT_PATH   = _ip or None
    TRUE_COUNT   = int(_tc) if _tc else None
    OUTPUT_TABLE = _ot or None
    print(f"Databricks widgets → input_path={INPUT_PATH}, true_count={TRUE_COUNT}, output_table={OUTPUT_TABLE}")
else:
    print("Not on Databricks — using local defaults from the config cell.")

## 2 · Environment bootstrap

Attach to whatever Spark session is already running (Databricks provides one; locally we start one). Resolve every parameter through the config → widget fallback so this same cell works in both places.

In [ ]:
import sys
from pathlib import Path

# Make src/ importable regardless of where the notebook is launched from.
# In Databricks Repos, ``Path.cwd()`` is the folder containing the notebook,
# so the same ``../src`` relative resolution works there and locally.
_PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_SRC = str(_PROJECT_ROOT / "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

from pyspark.sql import SparkSession

from wisdom_of_crowds.config import build_config

spark: SparkSession = SparkSession.builder.getOrCreate()

config = build_config(
    input_path_cfg=INPUT_PATH,
    true_count_cfg=TRUE_COUNT,
    output_table_cfg=OUTPUT_TABLE,
)

print(f"Spark version:  {spark.version}")
print(f"Experiment ID:  {EXPERIMENT_ID}")
print(f"Input path:     {config.input_path}")
print(f"True count:     {config.true_count}")
print(f"Output target:  {config.output_table}")

## 3 · Load the guesses

> *"About 800 tickets were issued, which were kindly lent me for examination after they had fulfilled their immediate purpose."*
>
> — Galton, *Vox Populi*

Read the CSV with an explicit schema — no schema inference — so drift becomes a loud failure rather than a silent one.

In [ ]:
from wisdom_of_crowds.transforms import load_guesses

raw = load_guesses(spark, config.input_path)
print(f"Loaded {raw.count()} rows from {config.input_path}")
raw.show(10, truncate=False)

## 4 · Flag defective or illegible rows

> *"After weeding thirteen cards out of the collection, as being defective or illegible, there remained 787 for discussion."*
>
> — Galton, *Vox Populi*

We annotate first, drop later. Each row gets an `array<string>` of reasons (empty when the row is valid) and a derived `is_defective` boolean. Extreme-but-legible guesses stay — Galton kept his fat tails and analysed them.

In [ ]:
from wisdom_of_crowds.transforms import flag_defective
from wisdom_of_crowds.columns import FlagColumns

flagged = flag_defective(raw)
flagged.select(
    "name", "guess",
    FlagColumns.DEFECT_REASONS, FlagColumns.IS_DEFECTIVE,
).show(10, truncate=False)

## 5 · Drop only the defective rows

One aggregation pass over the flagged frame gives us every count we need (total, dropped, per-reason). The cleaned frame drops the flag columns so nothing downstream leaks a defect reason into an analytical result.

In [ ]:
from wisdom_of_crowds.transforms import clean_guesses

clean = clean_guesses(flagged)
print(f"Rows in:      {clean.n_input}")
print(f"Rows kept:    {clean.n_kept}")
print(f"Rows dropped: {clean.n_dropped}")

if clean.n_dropped > 0:
    print("\nDropped by reason:")
    for reason, count in clean.dropped_by_reason.items():
        if count > 0:
            print(f"  {reason.value:22s} {count}")

## 6 · The crowd's answer — the middlemost estimate

> *"That conclusion is clearly not the average of all the estimates, which would give a voting power to 'cranks' in proportion to their crankiness… I wish to point out that the estimate to which least objection can be raised is the middlemost estimate."*
>
> — Galton, *One Vote, One Value*

In [ ]:
from wisdom_of_crowds.transforms import crowd_median

crowd_estimate = crowd_median(clean.kept)
print(f"The crowd's estimate (median of {clean.n_kept} guesses): {crowd_estimate}")

## 7 · Cross-check the median

The median from Spark should match the median computed independently in pure Python. If it disagrees, something is wrong and we stop before scoring.

In [ ]:
import statistics

pandas_guesses = clean.kept.select("guess").toPandas()["guess"].tolist()
independent_median = int(statistics.median(pandas_guesses))

print(f"Spark median:       {crowd_estimate}")
print(f"Pure-Python median: {independent_median}")

assert crowd_estimate == independent_median, (
    "Spark median disagrees with pure-Python median — investigate before continuing."
)
print("Cross-check passed.")

## 8 · Score the crowd against the truth

> *"Now the middlemost estimate is 1207 lb., and the weight of the dressed ox proved to be 1198 lb.; so the vox populi was in this case 9 lb., or 0.8 per cent, of the whole weight too high."*
>
> — Galton, *Vox Populi*

Signed error preserves direction (`+` too high, `−` too low) so "the crowd overshot" and "the crowd undershot" stay distinguishable.

In [ ]:
from wisdom_of_crowds.transforms import score_crowd

error = score_crowd(crowd_estimate=crowd_estimate, true_count=config.true_count)

print(f"Crowd estimate:  {error.crowd_estimate}")
print(f"True count:      {error.true_count}")
print(f"Signed error:    {error.signed_error:+d}")
print(f"Percent error:   {error.signed_percent_error:+.2f}%")

## 9 · Distribution of the guesses

A five-number summary — a condensed version of Galton's own centile table (*Vox Populi*, Nature 75:450). Symmetry — or the lack of it — is the point.

In [ ]:
from wisdom_of_crowds.transforms import summarise_distribution

dist = summarise_distribution(clean.kept)

print(f"n:      {dist.n}")
print(f"min:    {dist.minimum}")
print(f"Q1:     {dist.q1}")
print(f"median: {dist.median}")
print(f"Q3:     {dist.q3}")
print(f"max:    {dist.maximum}")
print(f"IQR:    {dist.iqr}")

## 10 · Persist the summary

On Databricks: a Delta table at `config.output_table` (when the value looks like `catalog.schema.table`).  
Locally, or on Databricks with a Volume path: Parquet at the same path.

We overwrite by `experiment_id` so re-running the notebook is idempotent.

In [ ]:
from wisdom_of_crowds.transforms import build_summary_row

summary_df = build_summary_row(
    spark=spark,
    experiment_id=EXPERIMENT_ID,
    error=error,
    distribution=dist,
    n_input=clean.n_input,
    n_dropped=clean.n_dropped,
)

# Heuristic: a `.` with no `/` looks like a table name; anything else is a path.
is_table_name = "/" not in config.output_table and "." in config.output_table

if is_table_name:
    (summary_df.write
        .format("delta").mode("overwrite")
        .option("replaceWhere", f"experiment_id = '{EXPERIMENT_ID}'")
        .saveAsTable(config.output_table))
else:
    (summary_df.write
        .mode("overwrite").parquet(config.output_table))

print(f"Wrote summary to: {config.output_table}")
summary_df.show(truncate=False, vertical=True)

## 11 · The verdict

In [ ]:
verdict_lines = [
    "───────────────────────────────────────────────",
    f"  Sweets-jar experiment · {EXPERIMENT_ID}",
    "───────────────────────────────────────────────",
    f"  Guesses submitted:     {clean.n_input}",
    f"  Defective/illegible:   {clean.n_dropped}",
    f"  Guesses used:          {dist.n}",
    "",
    f"  The crowd said:        {error.crowd_estimate}",
    f"  The jar actually held: {error.true_count}",
    f"  Off by:                {error.signed_error:+d}  ({error.signed_percent_error:+.2f}%)",
    "",
    f"  Spread:                min {dist.minimum}, Q1 {dist.q1}, "
    f"median {dist.median}, Q3 {dist.q3}, max {dist.maximum}",
    "───────────────────────────────────────────────",
]
print("\n".join(verdict_lines))